# Agent 与 Process 为什么必须分开？

## V0.7 Process Runtime / Scheduler / Accounting

这个 lab 让一个 Agent 的 Process 因预算超限被 BLOCKED，同时 Agent identity 仍然存在。

**Core:** Agent != Process. Budget exceeded != semantic failure.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Create Agent identity and Process runtime identity

In [ ]:
from agentkernel import (
    Agent, AgentBudget, CooperativeScheduler, ModelUsage,
    ProcessBudgetExceeded, ProcessState, SchedulerSafePoint, Session,
    UsageCollector,
)

agent = Agent.create(
    agent_id="lab-v0-7-agent",
    session=Session("lab-v0-7-session"),
    budget=AgentBudget(max_token_usage=5),
)
collector = UsageCollector()
scheduler = CooperativeScheduler(usage_collector=collector)
process = scheduler.create_process(process_id="lab-v0-7-process", agent=agent.control)
print_table([
    {"identity": "Agent", "id": agent.control.agent_id, "role": "capability principal"},
    {"identity": "Process", "id": process.process_id, "role": "schedulable runtime state"},
    {"identity": "Session", "id": agent.control.session_id, "role": "durable semantic journal"},
])
print_table([process_row(process)])

## 2. Dispatch and exceed budget at a safe point

In [ ]:
scheduler.dispatch(process.process_id)
collector.record_llm_usage(process.process_id, ModelUsage(input_tokens=4, output_tokens=2, total_tokens=6))
blocked = False
try:
    scheduler.safe_point(process.process_id, SchedulerSafePoint.AFTER_LLM_CALL)
except ProcessBudgetExceeded as error:
    blocked = True
    budget_error = str(error)
snapshot = collector.snapshot(process.process_id)
print_table([
    {"fact": "budget blocked", "value": blocked},
    {"fact": "process state", "value": process.state.value},
    {"fact": "observed tokens", "value": snapshot.token_usage},
    {"fact": "agent still exists", "value": agent.control.agent_id},
])

## 3. Host policy can reset runtime usage and unblock

In [ ]:
scheduler.reset_usage(process.process_id)
scheduler.unblock(process.process_id)
print_table([process_row(process)])
trajectory("READY", "RUNNING", "safe point observes budget", "BLOCKED", "usage reset by host policy", "READY")

## Invariant

Capability remains Agent-owned. Process state is runtime scheduling state and can be blocked without deleting the Agent.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- Process owns lifecycle and accounting state.
- Scheduler blocks at cooperative safe points.
- Agent identity remains the capability principal.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not implement preemptive scheduling.
- It does not make accounting a durable billing ledger.
- It does not use a real model provider.